Cell 1: data prep (light cleaning, subsample, tokenize)

In [1]:
import sys
sys.path.insert(0, '..')

import re
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from src.transformer.dataset import ArabicSentimentDataset
from src.utils.data_loader import load_twitter

URL_PATTERN = re.compile(r"http\S+|www\S+")
def light_clean(text: str) -> str:
    return URL_PATTERN.sub(" ", text).strip()

df = load_twitter()
label_map = {"Negative": 0, "Positive": 1}

train_df = df[df["split"] == "train"].copy()
test_df = df[df["split"] == "test"].copy()
train_df["text"] = train_df["text"].apply(light_clean)
test_df["text"] = test_df["text"].apply(light_clean)

print("Full train size:", len(train_df), "| Test size:", len(test_df))

tokenizer = AutoTokenizer.from_pretrained("aubmindlab/bert-base-arabertv2")

Full train size: 45275 | Test size: 11520


Cell 2: the model and the custom training loop

In [ ]:
import time
import os
from sklearn.model_selection import train_test_split
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score

train_final, val_df = train_test_split(
    train_df, test_size=0.1, stratify=train_df["label"], random_state=42
)
print("Train size:", len(train_final), "| Val size:", len(val_df), "| Test size:", len(test_df))

train_dataset = ArabicSentimentDataset(
    train_final["text"], train_final["label"].map(label_map), tokenizer, max_length=64
)
val_dataset = ArabicSentimentDataset(
    val_df["text"], val_df["label"].map(label_map), tokenizer, max_length=64
)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)

device = torch.device("cpu")
model = AutoModelForSequenceClassification.from_pretrained(
    "aubmindlab/bert-base-arabertv2", num_labels=2
)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

EPOCHS = 3
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
)

os.makedirs("../models", exist_ok=True)

for epoch in range(EPOCHS):
    epoch_start = time.time()
    model.train()
    total_loss = 0.0
    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

        if step % 200 == 0:
            elapsed = (time.time() - epoch_start) / 60
            print(f"  epoch {epoch+1} step {step}/{len(train_loader)} — loss {loss.item():.4f} — {elapsed:.1f} min elapsed")

    avg_train_loss = total_loss / len(train_loader)

    model.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)
            val_preds.extend(preds.cpu().tolist())
            val_labels.extend(labels.cpu().tolist())
    val_f1 = f1_score(val_labels, val_preds, average="macro")

    epoch_time = (time.time() - epoch_start) / 60
    print(f"Epoch {epoch+1}/{EPOCHS} done in {epoch_time:.1f} min — train loss {avg_train_loss:.4f} — val F1 macro {val_f1:.4f}")

    ckpt_dir = f"../models/arabert-twitter-full-epoch{epoch+1}"
    model.save_pretrained(ckpt_dir)
    tokenizer.save_pretrained(ckpt_dir)
    print(f"  saved checkpoint to {ckpt_dir}")

print("Training complete.")

Train size: 40747 | Val size: 4528 | Test size: 11520


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  epoch 1 step 0/5094 — loss 0.8270 — 0.1 min elapsed
  epoch 1 step 200/5094 — loss 0.8075 — 13.7 min elapsed
  epoch 1 step 400/5094 — loss 0.6748 — 25.9 min elapsed
  epoch 1 step 600/5094 — loss 0.7319 — 39.0 min elapsed
  epoch 1 step 800/5094 — loss 0.5309 — 51.2 min elapsed
  epoch 1 step 1000/5094 — loss 0.3991 — 63.2 min elapsed
  epoch 1 step 1200/5094 — loss 0.5278 — 75.7 min elapsed
  epoch 1 step 1400/5094 — loss 0.5481 — 88.3 min elapsed
  epoch 1 step 1600/5094 — loss 0.5291 — 101.6 min elapsed
  epoch 1 step 1800/5094 — loss 0.5907 — 113.7 min elapsed
  epoch 1 step 2000/5094 — loss 0.6252 — 125.9 min elapsed
  epoch 1 step 2200/5094 — loss 0.3427 — 138.0 min elapsed
  epoch 1 step 2400/5094 — loss 0.8565 — 150.8 min elapsed
  epoch 1 step 2600/5094 — loss 0.3911 — 163.7 min elapsed
  epoch 1 step 2800/5094 — loss 0.2985 — 176.0 min elapsed
  epoch 1 step 3000/5094 — loss 0.2998 — 188.2 min elapsed
  epoch 1 step 3200/5094 — loss 0.5985 — 200.5 min elapsed
  epoch 1 ste

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  saved checkpoint to ../models/arabert-twitter-full-epoch1
  epoch 2 step 0/5094 — loss 0.3789 — 0.2 min elapsed
  epoch 2 step 200/5094 — loss 0.4062 — 12.9 min elapsed
  epoch 2 step 400/5094 — loss 0.5589 — 25.1 min elapsed
  epoch 2 step 600/5094 — loss 0.4085 — 37.2 min elapsed
  epoch 2 step 800/5094 — loss 0.4081 — 49.2 min elapsed
  epoch 2 step 1000/5094 — loss 0.4359 — 62.4 min elapsed
  epoch 2 step 1200/5094 — loss 0.2454 — 75.9 min elapsed
  epoch 2 step 1400/5094 — loss 0.4155 — 87.7 min elapsed
  epoch 2 step 1600/5094 — loss 0.2004 — 99.7 min elapsed
  epoch 2 step 1800/5094 — loss 0.4462 — 111.4 min elapsed
  epoch 2 step 2000/5094 — loss 0.7457 — 125.2 min elapsed
  epoch 2 step 2200/5094 — loss 0.2568 — 138.0 min elapsed
  epoch 2 step 2400/5094 — loss 0.5737 — 150.1 min elapsed
  epoch 2 step 2600/5094 — loss 0.1461 — 162.2 min elapsed
  epoch 2 step 2800/5094 — loss 0.9442 — 174.4 min elapsed
  epoch 2 step 3000/5094 — loss 0.4081 — 189.6 min elapsed
  epoch 2 ste

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  saved checkpoint to ../models/arabert-twitter-full-epoch2
  epoch 3 step 0/5094 — loss 0.1598 — 0.1 min elapsed
  epoch 3 step 200/5094 — loss 0.1221 — 12.3 min elapsed
  epoch 3 step 400/5094 — loss 0.1862 — 24.4 min elapsed
  epoch 3 step 600/5094 — loss 0.1280 — 38.2 min elapsed
  epoch 3 step 800/5094 — loss 0.2854 — 50.4 min elapsed
  epoch 3 step 1000/5094 — loss 0.0184 — 62.4 min elapsed
  epoch 3 step 1200/5094 — loss 0.4541 — 74.6 min elapsed
  epoch 3 step 1400/5094 — loss 0.4985 — 86.6 min elapsed
  epoch 3 step 1600/5094 — loss 0.0861 — 100.8 min elapsed
  epoch 3 step 1800/5094 — loss 0.1140 — 112.8 min elapsed
  epoch 3 step 2000/5094 — loss 0.2782 — 125.7 min elapsed
  epoch 3 step 2200/5094 — loss 0.1845 — 137.8 min elapsed
  epoch 3 step 2400/5094 — loss 0.0669 — 150.9 min elapsed
  epoch 3 step 2600/5094 — loss 0.1743 — 165.2 min elapsed
  epoch 3 step 2800/5094 — loss 0.1168 — 177.3 min elapsed
  epoch 3 step 3000/5094 — loss 0.0555 — 189.5 min elapsed
  epoch 3 st